---
# US SuperStore Analysis
---

---
## Introduction
---

This is a Data Modeling file used to extract the data from the source(Kaggle), clean it with the help of python libraries and then send the clean data to other jupyter files(eeda and insights) for analytical insights and loading the cleaned data into cloud database(Neon) for storage and ACID compliance so as to share with the Bi tools like Sheets and Tableau.

Data Source: https://www.kaggle.com/datasets/rohitsahoo/sales-forecasting

---
## Python libraries
---

In [1]:
import numpy as np
import pandas as pd

---
## ETL (Extract, Transform & Load)
---

### Extract

#### Extracting into a DataFrame

In [2]:
df = pd.read_csv('/app/data/raw/train.csv')

#### Basic Info

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   str    
 2   Order Date     9800 non-null   str    
 3   Ship Date      9800 non-null   str    
 4   Ship Mode      9800 non-null   str    
 5   Customer ID    9800 non-null   str    
 6   Customer Name  9800 non-null   str    
 7   Segment        9800 non-null   str    
 8   Country        9800 non-null   str    
 9   City           9800 non-null   str    
 10  State          9800 non-null   str    
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   str    
 13  Product ID     9800 non-null   str    
 14  Category       9800 non-null   str    
 15  Sub-Category   9800 non-null   str    
 16  Product Name   9800 non-null   str    
 17  Sales          9800 non-null   float64
dtypes: float64(2), int6

In [4]:
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales'],
      dtype='str')

### Transform

#### snake-case column names

**This is done so it is easier to use column names in the code and also when this data would be loaded in a sql database the column names need not to be written in inveted commas and can be written normally.**

In [5]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [6]:
df.columns

Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'segment', 'country', 'city', 'state',
       'postal_code', 'region', 'product_id', 'category', 'sub-category',
       'product_name', 'sales'],
      dtype='str')

#### duplicates

**Originally no duplicates were found but after removing the row id column it is seen that there is one duplicated entry with order id US-2015-150119.**

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.drop('row_id', axis=1, inplace=True)

In [9]:
df.duplicated().sum()

np.int64(1)

In [10]:
df[df.duplicated(keep=False)]

,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub-category,product_name,sales
3405,US-2015-150119,23/04/2015,27/04/2015,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,Ohio,43229.0,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372
3406,US-2015-150119,23/04/2015,27/04/2015,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,Ohio,43229.0,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372


In [11]:
df.drop_duplicates(inplace=True)

In [12]:
df.info()

<class 'pandas.DataFrame'>
Index: 9799 entries, 0 to 9799
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       9799 non-null   str    
 1   order_date     9799 non-null   str    
 2   ship_date      9799 non-null   str    
 3   ship_mode      9799 non-null   str    
 4   customer_id    9799 non-null   str    
 5   customer_name  9799 non-null   str    
 6   segment        9799 non-null   str    
 7   country        9799 non-null   str    
 8   city           9799 non-null   str    
 9   state          9799 non-null   str    
 10  postal_code    9788 non-null   float64
 11  region         9799 non-null   str    
 12  product_id     9799 non-null   str    
 13  category       9799 non-null   str    
 14  sub-category   9799 non-null   str    
 15  product_name   9799 non-null   str    
 16  sales          9799 non-null   float64
dtypes: float64(2), str(15)
memory usage: 3.1 MB


#### Null Values

Null values were only found in postal code column and since it was only of rows which were from state Vermont and city Burlington in the east region of USA it was filled with the postal code 5401.

In [13]:
df.isna().sum()

order_id          0
order_date        0
ship_date         0
ship_mode         0
customer_id       0
customer_name     0
segment           0
country           0
city              0
state             0
postal_code      11
region            0
product_id        0
category          0
sub-category      0
product_name      0
sales             0
dtype: int64

In [14]:
df[df['postal_code'].isna()]

,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub-category,product_name,sales
2234,CA-2018-104066,05/12/2018,10/12/2018,Standard Class,QJ-19255,Quincy Jones,Corporate,United States,Burlington,Vermont,NaN,East,TEC-AC-10001013,Technology,Accessories,Logitech ClearChat Comfort/USB Headset H390,205.03
5274,CA-2016-162887,07/11/2016,09/11/2016,Second Class,SV-20785,Stewart Visinsky,Consumer,United States,Burlington,Vermont,NaN,East,FUR-CH-10000595,Furniture,Chairs,Safco Contoured Stacking Chairs,715.20
8798,US-2017-150140,06/04/2017,10/04/2017,Standard Class,VM-21685,Valerie Mitchum,Home Office,United States,Burlington,Vermont,NaN,East,TEC-PH-10002555,Technology,Phones,Nortel Meridian M5316 Digital phone,1294.75
9146,US-2017-165505,23/01/2017,27/01/2017,Standard Class,CB-12535,Claudia Bergmann,Corporate,United States,Burlington,Vermont,NaN,East,TEC-AC-10002926,Technology,Accessories,Logitech Wireless Marathon Mouse M705,99.98
9147,US-2017-165505,23/01/2017,27/01/2017,Standard Class,CB-12535,Claudia Bergmann,Corporate,United States,Burlington,Vermont,NaN,East,OFF-AR-10003477,Office Supplies,Art,4009 Highlighters,8.04
9148,US-2017-165505,23/01/2017,27/01/2017,Standard Class,CB-12535,Claudia Bergmann,Corporate,United States,Burlington,Vermont,NaN,East,OFF-ST-10001526,Office Supplies,Storage,Iceberg Mobile Mega Data/Printer Cart,1564.29
9386,US-2018-127292,19/01/2018,23/01/2018,Standard Class,RM-19375,Raymond Messe,Consumer,United States,Burlington,Vermont,NaN,East,OFF-PA-10000157,Office Supplies,Paper,Xerox 191,79.92
9387,US-2018-127292,19/01/2018,23/01/2018,Standard Class,RM-19375,Raymond Messe,Consumer,United States,Burlington,Vermont,NaN,East,OFF-PA-10001970,Office Supplies,Paper,Xerox 1881,12.28
9388,US-2018-127292,19/01/2018,23/01/2018,Standard Class,RM-19375,Raymond Messe,Consumer,United States,Burlington,Vermont,NaN,East,OFF-AP-10000828,Office Supplies,Appliances,Avanti 4.4 Cu. Ft. Refrigerator,542.94
9389,US-2018-127292,19/01/2018,23/01/2018,Standard Class,RM-19375,Raymond Messe,Consumer,United States,Burlington,Vermont,NaN,East,OFF-EN-10001509,Office Supplies,Envelopes,Poly String Tie Envelopes,2.04


In [15]:
df['postal_code'] = df['postal_code'].fillna(5401)

In [16]:
df['postal_code'] = df['postal_code'].astype('uint32')

In [17]:
df.isna().sum()

order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
sub-category     0
product_name     0
sales            0
dtype: int64

#### Transforming str dates into datetime dtypes

**Normal str date columns are converted into datetime dtype columns so as to simply trend analysis and data consistency.**

In [18]:
df.info()

<class 'pandas.DataFrame'>
Index: 9799 entries, 0 to 9799
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       9799 non-null   str    
 1   order_date     9799 non-null   str    
 2   ship_date      9799 non-null   str    
 3   ship_mode      9799 non-null   str    
 4   customer_id    9799 non-null   str    
 5   customer_name  9799 non-null   str    
 6   segment        9799 non-null   str    
 7   country        9799 non-null   str    
 8   city           9799 non-null   str    
 9   state          9799 non-null   str    
 10  postal_code    9799 non-null   uint32 
 11  region         9799 non-null   str    
 12  product_id     9799 non-null   str    
 13  category       9799 non-null   str    
 14  sub-category   9799 non-null   str    
 15  product_name   9799 non-null   str    
 16  sales          9799 non-null   float64
dtypes: float64(1), str(15), uint32(1)
memory usage: 3.0 MB


In [19]:
df['order_date'].head()

0    08/11/2017
1    08/11/2017
2    12/06/2017
3    11/10/2016
4    11/10/2016
Name: order_date, dtype: str

In [20]:
df['ship_date'].head()

0    11/11/2017
1    11/11/2017
2    16/06/2017
3    18/10/2016
4    18/10/2016
Name: ship_date, dtype: str

In [21]:
df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=True)
df['ship_date'] = pd.to_datetime(df['ship_date'], dayfirst=True)

In [22]:
df.info()

<class 'pandas.DataFrame'>
Index: 9799 entries, 0 to 9799
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_id       9799 non-null   str           
 1   order_date     9799 non-null   datetime64[us]
 2   ship_date      9799 non-null   datetime64[us]
 3   ship_mode      9799 non-null   str           
 4   customer_id    9799 non-null   str           
 5   customer_name  9799 non-null   str           
 6   segment        9799 non-null   str           
 7   country        9799 non-null   str           
 8   city           9799 non-null   str           
 9   state          9799 non-null   str           
 10  postal_code    9799 non-null   uint32        
 11  region         9799 non-null   str           
 12  product_id     9799 non-null   str           
 13  category       9799 non-null   str           
 14  sub-category   9799 non-null   str           
 15  product_name   9799 non-null   str   

#### Featured cols

##### related to order and ship dates

**Created a calculated column to better understand the shipping time.**

In [23]:
df['ship_mode'].value_counts()

ship_mode
Standard Class    5858
Second Class      1902
First Class       1501
Same Day           538
Name: count, dtype: int64

In [24]:
df['shipping_days'] = (df['ship_date'] - df['order_date']).dt.days

In [25]:
df[['order_date', 'ship_date', 'shipping_days']]

,order_date,ship_date,shipping_days
0,2017-11-08,2017-11-11,3
1,2017-11-08,2017-11-11,3
2,2017-06-12,2017-06-16,4
3,2016-10-11,2016-10-18,7
4,2016-10-11,2016-10-18,7
...,...,...,...
9795,2017-05-21,2017-05-28,7
9796,2016-01-12,2016-01-17,5
9797,2016-01-12,2016-01-17,5
9798,2016-01-12,2016-01-17,5


##### Creating a new ship mode col as older one had discrepencies

**The categorical column of ship mode did not match exactly with the dates so it was recreated to give more structure.**

In [26]:
df['ship_mode_new'] = pd.cut(
    df['shipping_days'],
    bins=[(-np.inf), 1, 3, 5, np.inf],
    labels=['Same Day', 'First Class', 'Second Class', 'Standard Class'],
    right=False
)

In [27]:
df['ship_mode_new'].value_counts()

ship_mode_new
Standard Class    3932
Second Class      3695
First Class       1658
Same Day           514
Name: count, dtype: int64

In [28]:
df.drop('ship_mode', axis=1, inplace=True)

In [29]:
df.rename(columns={
    'ship_mode_new' : 'ship_mode'
}, inplace=True)

In [30]:
df.info()

<class 'pandas.DataFrame'>
Index: 9799 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_id       9799 non-null   str           
 1   order_date     9799 non-null   datetime64[us]
 2   ship_date      9799 non-null   datetime64[us]
 3   customer_id    9799 non-null   str           
 4   customer_name  9799 non-null   str           
 5   segment        9799 non-null   str           
 6   country        9799 non-null   str           
 7   city           9799 non-null   str           
 8   state          9799 non-null   str           
 9   postal_code    9799 non-null   uint32        
 10  region         9799 non-null   str           
 11  product_id     9799 non-null   str           
 12  category       9799 non-null   str           
 13  sub-category   9799 non-null   str           
 14  product_name   9799 non-null   str           
 15  sales          9799 non-null   float6

#### Data Inconsistency

**The order id was inconsistent with the name scheme as some rows had CA as their prefix and others had US as their prefix, although in the given data United States is the only country name used so as to not create a confusion the order id column is recreated.**

In [31]:
df['country'].value_counts()

country
United States    9799
Name: count, dtype: int64

In [32]:
df[['order_id', 'country']]

,order_id,country
0,CA-2017-152156,United States
1,CA-2017-152156,United States
2,CA-2017-138688,United States
3,US-2016-108966,United States
4,US-2016-108966,United States
...,...,...
9795,CA-2017-125920,United States
9796,CA-2016-128608,United States
9797,CA-2016-128608,United States
9798,CA-2016-128608,United States


In [33]:
df['order_id_new'] = df['order_id']

In [34]:
df['order_id_new'] = df['order_id_new'].str.replace('CA', 'US')

In [35]:
df[df['order_id'].str.startswith('CA')]

,order_id,order_date,ship_date,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub-category,product_name,sales,shipping_days,ship_mode,order_id_new
0,CA-2017-152156,2017-11-08,2017-11-11,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.960,3,Second Class,US-2017-152156
1,CA-2017-152156,2017-11-08,2017-11-11,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.940,3,Second Class,US-2017-152156
2,CA-2017-138688,2017-06-12,2017-06-16,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.620,4,Second Class,US-2017-138688
5,CA-2015-115812,2015-06-09,2015-06-14,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.860,5,Standard Class,US-2015-115812
6,CA-2015-115812,2015-06-09,2015-06-14,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.280,5,Standard Class,US-2015-115812
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9795,CA-2017-125920,2017-05-21,2017-05-28,SH-19975,Sally Hughsby,Corporate,United States,Chicago,Illinois,60610,Central,OFF-BI-10003429,Office Supplies,Binders,"Cardinal HOLDit! Binder Insert Strips,Extra St...",3.798,7,Standard Class,US-2017-125920
9796,CA-2016-128608,2016-01-12,2016-01-17,CS-12490,Cindy Schnelling,Corporate,United States,Toledo,Ohio,43615,East,OFF-AR-10001374,Office Supplies,Art,"BIC Brite Liner Highlighters, Chisel Tip",10.368,5,Standard Class,US-2016-128608
9797,CA-2016-128608,2016-01-12,2016-01-17,CS-12490,Cindy Schnelling,Corporate,United States,Toledo,Ohio,43615,East,TEC-PH-10004977,Technology,Phones,GE 30524EE4,235.188,5,Standard Class,US-2016-128608
9798,CA-2016-128608,2016-01-12,2016-01-17,CS-12490,Cindy Schnelling,Corporate,United States,Toledo,Ohio,43615,East,TEC-PH-10000912,Technology,Phones,Anker 24W Portable Micro USB Car Charger,26.376,5,Standard Class,US-2016-128608


In [36]:
df[df['order_id_new'].str.startswith('CA')]

,order_id,order_date,ship_date,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub-category,product_name,sales,shipping_days,ship_mode,order_id_new


In [37]:
df.drop('order_id', axis=1, inplace=True)

In [38]:
df.rename(columns={
    'order_id_new' : 'order_id'
}, inplace=True)

In [39]:
df.duplicated().sum()

np.int64(0)

#### Fact & Dimension Table Creation

##### Product Dimension Table

In [40]:
# products = (df[['product_id', 'product_name', 'category', 'sub_category']].drop_duplicates().reset_index(drop=True).copy())

In [41]:
df.rename(columns={
    'sub-category' : 'sub_category'
}, inplace=True)

In [42]:
products = (df[['product_id', 'product_name', 'category', 'sub_category']].drop_duplicates().reset_index(drop=True).copy())

###### Basic Info

Basic Info

In [43]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 1893 entries, 0 to 1892
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_id    1893 non-null   str  
 1   product_name  1893 non-null   str  
 2   category      1893 non-null   str  
 3   sub_category  1893 non-null   str  
dtypes: str(4)
memory usage: 193.2 KB


In [44]:
products['product_id'].nunique()

1861

In [45]:
products['product_name'].nunique()

1849

In [46]:
products

,product_id,product_name,category,sub_category
0,FUR-BO-10001798,Bush Somerset Collection Bookcase,Furniture,Bookcases
1,FUR-CH-10000454,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Furniture,Chairs
2,OFF-LA-10000240,Self-Adhesive Address Labels for Typewriters b...,Office Supplies,Labels
3,FUR-TA-10000577,Bretford CR4500 Series Slim Rectangular Table,Furniture,Tables
4,OFF-ST-10000760,Eldon Fold 'N Roll Cart System,Office Supplies,Storage
...,...,...,...,...
1888,TEC-AC-10002380,Sony 8GB Class 10 Micro SDHC R40 Memory Card,Technology,Accessories
1889,TEC-PH-10002817,RCA ViSYS 25425RE1 Corded phone,Technology,Phones
1890,TEC-MA-10003589,Cisco 8961 IP Phone Charcoal,Technology,Machines
1891,OFF-AP-10003099,"Eureka Hand Vacuum, Bagless",Office Supplies,Appliances


###### Checks

Checks

In [47]:
product_check = (
    df.groupby("product_id")[
        ["product_name", "category", "sub_category"]
    ]
    .nunique()
)

product_check.gt(1).sum()

product_name    32
category         0
sub_category     0
dtype: int64

In [48]:
product_check[
    product_check.gt(1).any(axis=1)
]

,product_name,category,sub_category
product_id,,,
FUR-BO-10002213,2,1,1
FUR-CH-10001146,2,1,1
FUR-FU-10001473,2,1,1
FUR-FU-10004017,2,1,1
FUR-FU-10004091,2,1,1
FUR-FU-10004270,2,1,1
FUR-FU-10004848,2,1,1
FUR-FU-10004864,2,1,1
OFF-AP-10000576,2,1,1


In [49]:
problem_ids = (
    product_check[product_check["product_name"] > 1]
    .index
)

df[df["product_id"].isin(problem_ids)][
    ['product_id', 'product_name', 'category', 'sub_category']
].drop_duplicates().sort_values("product_id")

,product_id,product_name,category,sub_category
2115,FUR-BO-10002213,DMI Eclipse Executive Suite Bookcases,Furniture,Bookcases
2471,FUR-BO-10002213,"Sauder Forest Hills Library, Woodland Oak Finish",Furniture,Bookcases
66,FUR-CH-10001146,"Global Value Mid-Back Manager's Chair, Gray",Furniture,Chairs
128,FUR-CH-10001146,"Global Task Chair, Black",Furniture,Chairs
1459,FUR-FU-10001473,DAX Wood Document Frame,Furniture,Furnishings
...,...,...,...,...
3126,TEC-PH-10002200,Aastra 6757i CT Wireless VoIP phone,Technology,Phones
1378,TEC-PH-10002310,Panasonic KX T7731-B Digital phone,Technology,Phones
2596,TEC-PH-10002310,Plantronics Calisto P620-M USB Wireless Speake...,Technology,Phones
922,TEC-PH-10004531,OtterBox Commuter Series Case - iPhone 5 & 5s,Technology,Phones


In [50]:
print("Unique Product IDs:", df["product_id"].nunique())
print("Unique Product Names:", df["product_name"].nunique())
print(
    "Unique Product ID + Name:",
    df[["product_id", "product_name"]].drop_duplicates().shape[0]
)
print(
    "Unique Product ID + Name + Category + Subcategory:",
    df[['product_id', 
        'product_name', 
        'category', 
        'sub_category']].drop_duplicates().shape[0]
)

Unique Product IDs: 1861
Unique Product Names: 1849
Unique Product ID + Name: 1893
Unique Product ID + Name + Category + Subcategory: 1893


In [51]:
problem_ids = (
    product_check[product_check["product_name"] > 1]
    .index
)

df[df["product_id"].isin(problem_ids)][
    ['product_id', 'product_name', 'category', 'sub_category']
].drop_duplicates().sort_values("product_id")

,product_id,product_name,category,sub_category
2115,FUR-BO-10002213,DMI Eclipse Executive Suite Bookcases,Furniture,Bookcases
2471,FUR-BO-10002213,"Sauder Forest Hills Library, Woodland Oak Finish",Furniture,Bookcases
66,FUR-CH-10001146,"Global Value Mid-Back Manager's Chair, Gray",Furniture,Chairs
128,FUR-CH-10001146,"Global Task Chair, Black",Furniture,Chairs
1459,FUR-FU-10001473,DAX Wood Document Frame,Furniture,Furnishings
...,...,...,...,...
3126,TEC-PH-10002200,Aastra 6757i CT Wireless VoIP phone,Technology,Phones
1378,TEC-PH-10002310,Panasonic KX T7731-B Digital phone,Technology,Phones
2596,TEC-PH-10002310,Plantronics Calisto P620-M USB Wireless Speake...,Technology,Phones
922,TEC-PH-10004531,OtterBox Commuter Series Case - iPhone 5 & 5s,Technology,Phones


In [52]:
products['product_name'].unique()

<ArrowStringArray>
[                               'Bush Somerset Collection Bookcase',
      'Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back',
        'Self-Adhesive Address Labels for Typewriters by Universal',
                    'Bretford CR4500 Series Slim Rectangular Table',
                                   'Eldon Fold 'N Roll Cart System',
 'Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood',
                                                       'Newell 322',
                                   'Mitel 5320 IP Phone VoIP phone',
             'DXL Angle-View Binders with Locking Rings by Samsill',
                                 'Belkin F5C206VTEL 6 Outlet Surge',
 ...
     'Global Enterprise Series Seating Low-Back Swivel/Tilt Chairs',
  'Self-Adhesive Address Labels for Typewriters with Dispenser Box',
             'Office Star - Task Chair with Contemporary Loop Arms',
                                              'Grip Seal Envelopes',
          

###### Conclusion

Conclusion

There are 1849 unique product names and 1861 product id's which gets a mismatch of 11 id's matched to the same product name more than once, therefore we would be creating a new product id column with the same method as the old one adopting it's features.

In [53]:
products_dim = (df[["product_name", "category", "sub_category"]].drop_duplicates().reset_index(drop=True).copy())

In [54]:
products_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 1849 entries, 0 to 1848
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_name  1849 non-null   str  
 1   category      1849 non-null   str  
 2   sub_category  1849 non-null   str  
dtypes: str(3)
memory usage: 147.7 KB


In [55]:
product_name_check = (
    df.groupby("product_name")[["category", "sub_category"]]
      .nunique()
)

product_name_check.gt(1).sum()

category        0
sub_category    0
dtype: int64

Creating a surrogate key

In [56]:
products_dim.insert(0, 'product_key', range(1, len(products_dim) + 1))

Creating the actual product id with the same optics as the older product id

In [57]:
products_dim["product_id_n"] = (
    products_dim["category"].str[:3].str.upper()
    + "-"
    + products_dim["sub_category"].str[:2].str.upper()
    + "-"
    + products_dim["product_key"].astype(str).str.zfill(8)
)

In [58]:
products_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 1849 entries, 0 to 1848
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_key   1849 non-null   int64
 1   product_name  1849 non-null   str  
 2   category      1849 non-null   str  
 3   sub_category  1849 non-null   str  
 4   product_id_n  1849 non-null   str  
dtypes: int64(1), str(4)
memory usage: 203.7 KB


Adding the corrected id back to df

In [59]:
df = pd.merge(df, 
              products_dim[['product_id_n', 'product_name', 'category', 'sub_category']],
              on=['product_name', 'category', 'sub_category'], 
              how='left', 
              validate='many_to_one')

In [60]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9799 entries, 0 to 9798
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_date     9799 non-null   datetime64[us]
 1   ship_date      9799 non-null   datetime64[us]
 2   customer_id    9799 non-null   str           
 3   customer_name  9799 non-null   str           
 4   segment        9799 non-null   str           
 5   country        9799 non-null   str           
 6   city           9799 non-null   str           
 7   state          9799 non-null   str           
 8   postal_code    9799 non-null   uint32        
 9   region         9799 non-null   str           
 10  product_id     9799 non-null   str           
 11  category       9799 non-null   str           
 12  sub_category   9799 non-null   str           
 13  product_name   9799 non-null   str           
 14  sales          9799 non-null   float64       
 15  shipping_days  9799 non-null   i

In [61]:
df['product_id'].nunique()

1861

In [62]:
df['product_id_n'].nunique()

1849

In [63]:
df.drop(columns='product_id', inplace=True)

In [64]:
df.rename(columns={
    'product_id_n' : 'product_id'
    }, inplace=True)

In [65]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9799 entries, 0 to 9798
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_date     9799 non-null   datetime64[us]
 1   ship_date      9799 non-null   datetime64[us]
 2   customer_id    9799 non-null   str           
 3   customer_name  9799 non-null   str           
 4   segment        9799 non-null   str           
 5   country        9799 non-null   str           
 6   city           9799 non-null   str           
 7   state          9799 non-null   str           
 8   postal_code    9799 non-null   uint32        
 9   region         9799 non-null   str           
 10  category       9799 non-null   str           
 11  sub_category   9799 non-null   str           
 12  product_name   9799 non-null   str           
 13  sales          9799 non-null   float64       
 14  shipping_days  9799 non-null   int64         
 15  ship_mode      9799 non-null   c

##### Orders Fact Table

In [66]:
df.columns

Index(['order_date', 'ship_date', 'customer_id', 'customer_name', 'segment',
       'country', 'city', 'state', 'postal_code', 'region', 'category',
       'sub_category', 'product_name', 'sales', 'shipping_days', 'ship_mode',
       'order_id', 'product_id'],
      dtype='str')

In [67]:
orders_fact = df[['order_id', 'order_date', 'ship_date', 'ship_mode', 'shipping_days', 
                  'customer_id', 'customer_name', 'segment', 
                  'country', 'region', 'state', 'city', 'postal_code', 
                  'product_id', 'product_name', 'category', 'sub_category', 
                  'sales'
                  ]].drop_duplicates().dropna().reset_index(drop=True).copy()

In [68]:
orders_fact.insert(0, 'orders_key', range(1, len(orders_fact) + 1))

In [69]:
orders_fact.info()

<class 'pandas.DataFrame'>
RangeIndex: 9799 entries, 0 to 9798
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   orders_key     9799 non-null   int64         
 1   order_id       9799 non-null   str           
 2   order_date     9799 non-null   datetime64[us]
 3   ship_date      9799 non-null   datetime64[us]
 4   ship_mode      9799 non-null   category      
 5   shipping_days  9799 non-null   int64         
 6   customer_id    9799 non-null   str           
 7   customer_name  9799 non-null   str           
 8   segment        9799 non-null   str           
 9   country        9799 non-null   str           
 10  region         9799 non-null   str           
 11  state          9799 non-null   str           
 12  city           9799 non-null   str           
 13  postal_code    9799 non-null   uint32        
 14  product_id     9799 non-null   str           
 15  product_name   9799 non-null   s

##### Creating Date Dimension Table

###### Concatinating 2 date cols into one

Concatinating dates from 2 cols into one

In [70]:
dates = pd.concat([df['order_date'], df['ship_date']], axis=0).dropna().drop_duplicates().sort_values().reset_index(drop=True)

In [71]:
dates

0      2015-01-03
1      2015-01-04
2      2015-01-05
3      2015-01-06
4      2015-01-07
          ...    
1426   2019-01-01
1427   2019-01-02
1428   2019-01-03
1429   2019-01-04
1430   2019-01-05
Length: 1431, dtype: datetime64[us]

###### Inserting into a DataFrame

Inserting into a DataFrame

In [72]:
date_dim = pd.DataFrame({
    'full_date' : dates
}).reset_index(drop=True)

In [73]:
date_dim

,full_date
0,2015-01-03
1,2015-01-04
2,2015-01-05
3,2015-01-06
4,2015-01-07
...,...
1426,2019-01-01
1427,2019-01-02
1428,2019-01-03
1429,2019-01-04


In [74]:
date_dim['date_key'] = range(1, len(date_dim) + 1)

###### Adding a surrogate key

Adding a surrogate key

In [75]:
date_dim = date_dim[['date_key', 'full_date']]

In [76]:
date_dim

,date_key,full_date
0,1,2015-01-03
1,2,2015-01-04
2,3,2015-01-05
3,4,2015-01-06
4,5,2015-01-07
...,...,...
1426,1427,2019-01-01
1427,1428,2019-01-02
1428,1429,2019-01-03
1429,1430,2019-01-04


###### featured cols related to the original dates

featured cols related to the original dates

In [77]:
# Calendar / normal date attributes
date_dim["day"] = date_dim["full_date"].dt.day
date_dim["day_name"] = date_dim["full_date"].dt.day_name()

date_dim["month"] = date_dim["full_date"].dt.month
date_dim["month_name"] = date_dim["full_date"].dt.month_name()

date_dim["quarter"] = date_dim["full_date"].dt.quarter
date_dim["year"] = date_dim["full_date"].dt.year

date_dim["day_of_year"] = date_dim["full_date"].dt.dayofyear

# Normal / calendar week
date_dim["week"] = date_dim["full_date"].dt.strftime("%U").astype(int)

# ISO date attributes
iso = date_dim["full_date"].dt.isocalendar()

date_dim["iso_year"] = iso.year
date_dim["iso_week"] = iso.week
date_dim["iso_day"] = iso.day

# Other useful attributes
date_dim["is_weekend"] = date_dim["full_date"].dt.dayofweek >= 5

date_dim["iso_year_week"] = (
    date_dim["iso_year"].astype(str)
    + "-"
    + date_dim["iso_week"].astype(str).str.zfill(2)
)

In [78]:
date_dim

,date_key,full_date,day,day_name,month,month_name,quarter,year,day_of_year,week,iso_year,iso_week,iso_day,is_weekend,iso_year_week
0,1,2015-01-03,3,Saturday,1,January,1,2015,3,0,2015,1,6,True,2015-01
1,2,2015-01-04,4,Sunday,1,January,1,2015,4,1,2015,1,7,True,2015-01
2,3,2015-01-05,5,Monday,1,January,1,2015,5,1,2015,2,1,False,2015-02
3,4,2015-01-06,6,Tuesday,1,January,1,2015,6,1,2015,2,2,False,2015-02
4,5,2015-01-07,7,Wednesday,1,January,1,2015,7,1,2015,2,3,False,2015-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1426,1427,2019-01-01,1,Tuesday,1,January,1,2019,1,0,2019,1,2,False,2019-01
1427,1428,2019-01-02,2,Wednesday,1,January,1,2019,2,0,2019,1,3,False,2019-01
1428,1429,2019-01-03,3,Thursday,1,January,1,2019,3,0,2019,1,4,False,2019-01
1429,1430,2019-01-04,4,Friday,1,January,1,2019,4,0,2019,1,5,False,2019-01


In [79]:
date_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 1431 entries, 0 to 1430
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date_key       1431 non-null   int64         
 1   full_date      1431 non-null   datetime64[us]
 2   day            1431 non-null   int32         
 3   day_name       1431 non-null   str           
 4   month          1431 non-null   int32         
 5   month_name     1431 non-null   str           
 6   quarter        1431 non-null   int32         
 7   year           1431 non-null   int32         
 8   day_of_year    1431 non-null   int32         
 9   week           1431 non-null   int64         
 10  iso_year       1431 non-null   UInt32        
 11  iso_week       1431 non-null   UInt32        
 12  iso_day        1431 non-null   UInt32        
 13  is_weekend     1431 non-null   bool          
 14  iso_year_week  1431 non-null   str           
dtypes: UInt32(3), bool(1), datetime6

##### Creating Ship mode Dimension Table

In [80]:
ship_mode_dim = df[['ship_mode']].drop_duplicates().reset_index(drop=True).copy()

In [81]:
ship_mode_dim.insert(0, 'ship_mode_key', range(1, len(ship_mode_dim) + 1))

In [82]:
ship_mode_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   ship_mode_key  4 non-null      int64   
 1   ship_mode      4 non-null      category
dtypes: category(1), int64(1)
memory usage: 417.0 bytes


##### Creating Customers Dimension Table

In [83]:
customers_dim = (df[['customer_id', 'customer_name', 'segment']].drop_duplicates().reset_index(drop=True).copy())

In [84]:
customers_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 793 entries, 0 to 792
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   customer_id    793 non-null    str  
 1   customer_name  793 non-null    str  
 2   segment        793 non-null    str  
dtypes: str(3)
memory usage: 42.1 KB


In [85]:
customers_dim.insert(0, 'customer_key', range(1, len(customers_dim) + 1))

In [86]:
customers_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 793 entries, 0 to 792
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   customer_key   793 non-null    int64
 1   customer_id    793 non-null    str  
 2   customer_name  793 non-null    str  
 3   segment        793 non-null    str  
dtypes: int64(1), str(3)
memory usage: 48.3 KB


In [87]:
customers_dim['customer_id'].nunique()

793

In [88]:
customers_dim['customer_name'].nunique()

793

In [89]:
df.groupby("customer_id")["customer_name"].nunique().value_counts()

customer_name
1    793
Name: count, dtype: int64

##### Creating Location Dimension Table

In [90]:
location_dim = df[['country', 'region', 'state', 'city', 'postal_code']].drop_duplicates().reset_index(drop=True).copy()

In [91]:
location_dim.insert(0, 'location_key', range(1, len(location_dim) + 1))

In [92]:
location_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 628 entries, 0 to 627
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   location_key  628 non-null    int64 
 1   country       628 non-null    str   
 2   region        628 non-null    str   
 3   state         628 non-null    str   
 4   city          628 non-null    str   
 5   postal_code   628 non-null    uint32
dtypes: int64(1), str(4), uint32(1)
memory usage: 49.0 KB


#### Merging dim tables with fact table

##### Orders fact table with date dim for order and ship date key

In [93]:
orders_fact = orders_fact.merge(date_dim[['date_key', 'full_date']], 
                                left_on='order_date', 
                                right_on='full_date', 
                                how='left',
                                validate='many_to_one'
                                )

In [94]:
orders_fact.rename(columns={'date_key' : 'order_date_key'}, inplace=True)

In [95]:
orders_fact.drop('full_date', axis=1, inplace=True)

In [96]:
orders_fact = orders_fact.merge(date_dim[['date_key', 'full_date']], 
                                left_on='ship_date', 
                                right_on='full_date', 
                                how='left', 
                                validate='many_to_one'
                                )

In [97]:
orders_fact.rename(columns={'date_key' : 'ship_date_key'}, inplace=True)

In [98]:
orders_fact.drop('full_date', axis=1, inplace=True)

In [99]:
orders_fact.info()

<class 'pandas.DataFrame'>
RangeIndex: 9799 entries, 0 to 9798
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   orders_key      9799 non-null   int64         
 1   order_id        9799 non-null   str           
 2   order_date      9799 non-null   datetime64[us]
 3   ship_date       9799 non-null   datetime64[us]
 4   ship_mode       9799 non-null   category      
 5   shipping_days   9799 non-null   int64         
 6   customer_id     9799 non-null   str           
 7   customer_name   9799 non-null   str           
 8   segment         9799 non-null   str           
 9   country         9799 non-null   str           
 10  region          9799 non-null   str           
 11  state           9799 non-null   str           
 12  city            9799 non-null   str           
 13  postal_code     9799 non-null   uint32        
 14  product_id      9799 non-null   str           
 15  product_name   

In [100]:
orders_fact.drop(columns=['order_date', 'ship_date'], inplace=True)

##### Orders fact table with products dim table

In [101]:
products_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 1849 entries, 0 to 1848
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_key   1849 non-null   int64
 1   product_name  1849 non-null   str  
 2   category      1849 non-null   str  
 3   sub_category  1849 non-null   str  
 4   product_id_n  1849 non-null   str  
dtypes: int64(1), str(4)
memory usage: 203.7 KB


In [102]:
products_dim.rename(columns={
    'product_id_n' : 'product_id'
}, inplace=True)

In [103]:
products_dim = products_dim[['product_key', 'product_id', 'product_name', 'category', 'sub_category']]

In [104]:
orders_fact = orders_fact.merge(products_dim[['product_key', 'product_id']], 
                                left_on='product_id', 
                                right_on='product_id', 
                                how='left',
                                validate='many_to_one'
                                )

In [105]:
orders_fact.info()

<class 'pandas.DataFrame'>
RangeIndex: 9799 entries, 0 to 9798
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   orders_key      9799 non-null   int64   
 1   order_id        9799 non-null   str     
 2   ship_mode       9799 non-null   category
 3   shipping_days   9799 non-null   int64   
 4   customer_id     9799 non-null   str     
 5   customer_name   9799 non-null   str     
 6   segment         9799 non-null   str     
 7   country         9799 non-null   str     
 8   region          9799 non-null   str     
 9   state           9799 non-null   str     
 10  city            9799 non-null   str     
 11  postal_code     9799 non-null   uint32  
 12  product_id      9799 non-null   str     
 13  product_name    9799 non-null   str     
 14  category        9799 non-null   str     
 15  sub_category    9799 non-null   str     
 16  sales           9799 non-null   float64 
 17  order_date_key  9799 non-

In [106]:
orders_fact['product_id'].nunique()

1849

In [107]:
orders_fact.drop(columns=['product_id', 'product_name', 'category', 'sub_category'], inplace=True)

In [108]:
orders_fact.info()

<class 'pandas.DataFrame'>
RangeIndex: 9799 entries, 0 to 9798
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   orders_key      9799 non-null   int64   
 1   order_id        9799 non-null   str     
 2   ship_mode       9799 non-null   category
 3   shipping_days   9799 non-null   int64   
 4   customer_id     9799 non-null   str     
 5   customer_name   9799 non-null   str     
 6   segment         9799 non-null   str     
 7   country         9799 non-null   str     
 8   region          9799 non-null   str     
 9   state           9799 non-null   str     
 10  city            9799 non-null   str     
 11  postal_code     9799 non-null   uint32  
 12  sales           9799 non-null   float64 
 13  order_date_key  9799 non-null   int64   
 14  ship_date_key   9799 non-null   int64   
 15  product_key     9799 non-null   int64   
dtypes: category(1), float64(1), int64(5), str(8), uint32(1)
memory usage: 1

###### Featured Cols

Adding featured cols for better understanding of data

In [109]:
product_metrics = (
    orders_fact
    .groupby("product_key")
    .agg(
        product_order_count=("order_id", "nunique"),
        product_sales=("sales", "sum")
    )
    .reset_index()
)

In [110]:
products_dim = products_dim.merge(product_metrics,
                                  on='product_key', 
                                  how='left',
                                  validate='one_to_one')

In [111]:
products_dim.head()

,product_key,product_id,product_name,category,sub_category,product_order_count,product_sales
0,1,FUR-BO-00000001,Bush Somerset Collection Bookcase,Furniture,Bookcases,4,1263.957
1,2,FUR-CH-00000002,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Furniture,Chairs,12,10637.528
2,3,OFF-LA-00000003,Self-Adhesive Address Labels for Typewriters b...,Office Supplies,Labels,7,153.510
3,4,FUR-TA-00000004,Bretford CR4500 Series Slim Rectangular Table,Furniture,Tables,8,7242.768
4,5,OFF-ST-00000005,Eldon Fold 'N Roll Cart System,Office Supplies,Storage,5,301.968


In [112]:
products_dim.describe().round(2)

,product_key,product_order_count,product_sales
count,1849.0,1849.00,1849.00
mean,925.0,5.30,1222.96
std,533.9,3.09,2774.00
min,1.0,1.00,1.62
25%,463.0,3.00,96.65
50%,925.0,5.00,299.97
75%,1387.0,7.00,1184.03
max,1849.0,47.00,61599.82


In [113]:
products_dim['order_count_cat'] = pd.cut(
    products_dim['product_order_count'],
    bins=[-np.inf, 10, 25, np.inf],
    labels=['low', 'moderate', 'high'],
    right=True
)

In [114]:
products_dim['sales_cat'] = pd.cut(
    products_dim['product_sales'],
    bins=[-float('inf'), 9999, 49999, float('inf')],
    labels=['low', 'moderate', 'high'],
    right=False
)

In [115]:
products_dim["product_value"] = np.select(
    [
        (products_dim["order_count_cat"] == "high") &
        (products_dim["sales_cat"] == "high"),

        (products_dim["order_count_cat"] == "low") &
        (products_dim["sales_cat"] == "low"),

        (products_dim["order_count_cat"] == "high") &
        (products_dim["sales_cat"] == "moderate"),

        (products_dim["order_count_cat"] == "low") &
        (products_dim["sales_cat"] == "moderate"),

        (products_dim["order_count_cat"] == "moderate") &
        (products_dim["sales_cat"] == "high"),

        (products_dim["order_count_cat"] == "moderate") &
        (products_dim["sales_cat"] == "low")
    ],
    [
        "very_high",
        "very_low",
        "high",
        "low",
        "high",
        "low"
    ],
    default="moderate"
)

In [116]:
products_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 1849 entries, 0 to 1848
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   product_key          1849 non-null   int64   
 1   product_id           1849 non-null   str     
 2   product_name         1849 non-null   str     
 3   category             1849 non-null   str     
 4   sub_category         1849 non-null   str     
 5   product_order_count  1849 non-null   int64   
 6   product_sales        1849 non-null   float64 
 7   order_count_cat      1849 non-null   category
 8   sales_cat            1849 non-null   category
 9   product_value        1849 non-null   str     
dtypes: category(2), float64(1), int64(2), str(5)
memory usage: 265.0 KB


##### Orders fact table with customers dim table

In [117]:
orders_fact = orders_fact.merge(customers_dim[['customer_key', 'customer_id', 'customer_name', 'segment']],
                        on=['customer_id', 'customer_name', 'segment'], 
                        how='left', 
                        validate='many_to_one'
                        )

In [118]:
orders_fact.info()

<class 'pandas.DataFrame'>
RangeIndex: 9799 entries, 0 to 9798
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   orders_key      9799 non-null   int64   
 1   order_id        9799 non-null   str     
 2   ship_mode       9799 non-null   category
 3   shipping_days   9799 non-null   int64   
 4   customer_id     9799 non-null   str     
 5   customer_name   9799 non-null   str     
 6   segment         9799 non-null   str     
 7   country         9799 non-null   str     
 8   region          9799 non-null   str     
 9   state           9799 non-null   str     
 10  city            9799 non-null   str     
 11  postal_code     9799 non-null   uint32  
 12  sales           9799 non-null   float64 
 13  order_date_key  9799 non-null   int64   
 14  ship_date_key   9799 non-null   int64   
 15  product_key     9799 non-null   int64   
 16  customer_key    9799 non-null   int64   
dtypes: category(1), float64(1

In [119]:
orders_fact.drop(columns=['customer_id', 'customer_name', 'segment'], inplace=True)

###### featured Cols

Adding featured cols for better understanding of data

In [120]:
customer_metrics = (
    orders_fact
    .groupby("customer_key")
    .agg(
        customer_order_count=("order_id", "nunique"),
        customer_sales=("sales", "sum"),
        avg_order_value=("sales", "mean")
    )
    .reset_index()
)

In [121]:
customers_dim = customers_dim.merge(
    customer_metrics,
    on="customer_key",
    how="left",
    validate='one_to_one'
)

In [122]:
customers_dim

,customer_key,customer_id,customer_name,segment,customer_order_count,customer_sales,avg_order_value
0,1,CG-12520,Claire Gute,Consumer,3,1148.7800,229.756000
1,2,DV-13045,Darrin Van Huff,Corporate,5,1119.4830,124.387000
2,3,SO-20335,Sean O'Donnell,Consumer,6,2602.5755,173.505033
3,4,BH-11710,Brosina Hoffman,Consumer,8,6255.3510,260.639625
4,5,AA-10480,Andrew Allen,Consumer,4,1790.5120,149.209333
...,...,...,...,...,...,...,...
788,789,CJ-11875,Carl Jackson,Corporate,1,16.5200,16.520000
789,790,RS-19870,Roy Skaria,Home Office,2,22.3280,7.442667
790,791,SC-20845,Sung Chung,Consumer,1,14.1120,14.112000
791,792,RE-19405,Ricardo Emerson,Consumer,1,48.3600,48.360000


In [123]:
customers_dim.describe().round(2)

,customer_key,customer_order_count,customer_sales,avg_order_value
count,793.00,793.00,793.00,793.00
mean,397.00,6.21,2851.52,227.40
std,229.06,2.53,2620.40,190.42
min,1.00,1.00,4.83,2.42
25%,199.00,4.00,1081.47,115.91
50%,397.00,6.00,2215.00,184.05
75%,595.00,8.00,3670.26,282.61
max,793.00,17.00,25043.05,1751.29


In [124]:
customers_dim['order_count_cat'] = pd.cut(
    customers_dim['customer_order_count'],
    bins=[-np.inf, 5, 10, np.inf],
    labels=['low', 'moderate', 'high'],
    right=True
)

In [125]:
customers_dim['sales_cat'] = pd.cut(
    customers_dim['customer_sales'],
    bins=[-float("inf"), 999, 9999, float("inf")],
    labels=['low', 'moderate', 'high'],
    right=True
)

In [126]:
customers_dim['avg_order_val_cat'] = pd.cut(
    customers_dim['avg_order_value'],
    bins=[-float("inf"), 499, 999, float("inf")],
    labels=['low', 'moderate', 'high'],
    right=True
)

In [127]:
score_map = {
    "low": 1,
    "moderate": 2,
    "high": 3
}

customers_dim["customer_value_score"] = (
    customers_dim["order_count_cat"].map(score_map).astype("int64")
    + customers_dim["avg_order_val_cat"].map(score_map).astype("int64")
    + customers_dim["sales_cat"].map(score_map).astype("int64")
)

In [128]:
customers_dim["customer_value_score"].dtype

dtype('int64')

In [129]:
customers_dim["customer_value"] = np.select(
    [
        customers_dim["customer_value_score"] == 3,
        customers_dim["customer_value_score"].isin([4, 5]),
        customers_dim["customer_value_score"] == 6,
        customers_dim["customer_value_score"].isin([7, 8]),
        customers_dim["customer_value_score"] == 9
    ],
    [
        "very_low",
        "low",
        "moderate",
        "high",
        "very_high"
    ],
    default="moderate"
)

In [130]:
customers_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 793 entries, 0 to 792
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   customer_key          793 non-null    int64   
 1   customer_id           793 non-null    str     
 2   customer_name         793 non-null    str     
 3   segment               793 non-null    str     
 4   customer_order_count  793 non-null    int64   
 5   customer_sales        793 non-null    float64 
 6   avg_order_value       793 non-null    float64 
 7   order_count_cat       793 non-null    category
 8   sales_cat             793 non-null    category
 9   avg_order_val_cat     793 non-null    category
 10  customer_value_score  793 non-null    int64   
 11  customer_value        793 non-null    str     
dtypes: category(3), float64(2), int64(3), str(4)
memory usage: 85.4 KB


In [131]:
customers_dim.drop(columns='customer_value_score', inplace=True)

##### Orders fact table with location dim table

In [132]:
orders_fact = orders_fact.merge(location_dim, 
                                on=['country', 'region', 'state', 'city', 'postal_code'], 
                                how='left', 
                                validate='many_to_one'
                                )

In [133]:
orders_fact.info()

<class 'pandas.DataFrame'>
RangeIndex: 9799 entries, 0 to 9798
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   orders_key      9799 non-null   int64   
 1   order_id        9799 non-null   str     
 2   ship_mode       9799 non-null   category
 3   shipping_days   9799 non-null   int64   
 4   country         9799 non-null   str     
 5   region          9799 non-null   str     
 6   state           9799 non-null   str     
 7   city            9799 non-null   str     
 8   postal_code     9799 non-null   uint32  
 9   sales           9799 non-null   float64 
 10  order_date_key  9799 non-null   int64   
 11  ship_date_key   9799 non-null   int64   
 12  product_key     9799 non-null   int64   
 13  customer_key    9799 non-null   int64   
 14  location_key    9799 non-null   int64   
dtypes: category(1), float64(1), int64(7), str(5), uint32(1)
memory usage: 1.5 MB


In [134]:
orders_fact.drop(columns=['country', 'region', 'state', 'city', 'postal_code'], inplace=True)

In [135]:
orders_fact.info()

<class 'pandas.DataFrame'>
RangeIndex: 9799 entries, 0 to 9798
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   orders_key      9799 non-null   int64   
 1   order_id        9799 non-null   str     
 2   ship_mode       9799 non-null   category
 3   shipping_days   9799 non-null   int64   
 4   sales           9799 non-null   float64 
 5   order_date_key  9799 non-null   int64   
 6   ship_date_key   9799 non-null   int64   
 7   product_key     9799 non-null   int64   
 8   customer_key    9799 non-null   int64   
 9   location_key    9799 non-null   int64   
dtypes: category(1), float64(1), int64(7), str(1)
memory usage: 834.1 KB


###### Featured Cols

Adding featured cols for better understanding of data

In [136]:
location_metrics = (
    orders_fact
    .groupby("location_key")
    .agg(
        total_orders=('order_id', 'nunique'),
        total_sales=('sales', 'sum')
    )
    .reset_index()
)

In [137]:
location_dim = pd.merge(
    location_dim,
    location_metrics,
    on='location_key',
    how='inner',
    validate='one_to_one'
)

In [138]:
location_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 628 entries, 0 to 627
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   location_key  628 non-null    int64  
 1   country       628 non-null    str    
 2   region        628 non-null    str    
 3   state         628 non-null    str    
 4   city          628 non-null    str    
 5   postal_code   628 non-null    uint32 
 6   total_orders  628 non-null    int64  
 7   total_sales   628 non-null    float64
dtypes: float64(1), int64(2), str(4), uint32(1)
memory usage: 58.9 KB


In [139]:
location_dim.describe().round(2)

,location_key,postal_code,total_orders,total_sales
count,628.00,628.00,628.00,628.00
mean,314.50,56146.49,7.84,3600.73
std,181.43,29183.49,15.13,7941.82
min,1.00,1040.00,1.00,1.39
25%,157.75,33018.75,2.00,264.65
50%,314.50,57551.00,3.00,1042.32
75%,471.25,83226.00,6.25,3265.86
max,628.00,99301.00,120.00,78534.04


In [140]:
location_dim['total_orders_cat'] = pd.cut(
    location_dim['total_orders'],
    bins=[-np.inf, 50, 100, np.inf],
    labels=['low', 'moderate', 'high'],
    right=True
)

In [141]:
location_dim['total_sales_cat'] = pd.cut(
    location_dim['total_sales'],
    bins=[-float('inf'), 9999, 49999, float('inf')],
    labels=['low', 'moderate', 'high'],
    right=True
)

In [142]:
location_dim["location_value"] = np.select(
    [
        (location_dim["total_orders_cat"] == "high") &
        (location_dim["total_sales_cat"] == "high"),

        (location_dim["total_orders_cat"] == "low") &
        (location_dim["total_sales_cat"] == "low"),

        (location_dim["total_orders_cat"] == "high") &
        (location_dim["total_sales_cat"] == "moderate"),

        (location_dim["total_orders_cat"] == "low") &
        (location_dim["total_sales_cat"] == "moderate"),

        (location_dim["total_orders_cat"] == "moderate") &
        (location_dim["total_sales_cat"] == "high"),

        (location_dim["total_orders_cat"] == "moderate") &
        (location_dim["total_sales_cat"] == "low")
    ],
    [
        "very_high",
        "very_low",
        "high",
        "low",
        "high",
        "low"
    ],
    default="moderate"
)

##### Orders fact table with Shipping Mode dim table

In [143]:
orders_fact = orders_fact.merge(ship_mode_dim, 
                                on='ship_mode', 
                                how='left',
                                validate='many_to_one'
                                )

In [144]:
orders_fact.info()

<class 'pandas.DataFrame'>
RangeIndex: 9799 entries, 0 to 9798
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   orders_key      9799 non-null   int64   
 1   order_id        9799 non-null   str     
 2   ship_mode       9799 non-null   category
 3   shipping_days   9799 non-null   int64   
 4   sales           9799 non-null   float64 
 5   order_date_key  9799 non-null   int64   
 6   ship_date_key   9799 non-null   int64   
 7   product_key     9799 non-null   int64   
 8   customer_key    9799 non-null   int64   
 9   location_key    9799 non-null   int64   
 10  ship_mode_key   9799 non-null   int64   
dtypes: category(1), float64(1), int64(8), str(1)
memory usage: 910.7 KB


In [145]:
orders_fact.drop('ship_mode', axis=1, inplace=True)

In [146]:
orders_fact.info()

<class 'pandas.DataFrame'>
RangeIndex: 9799 entries, 0 to 9798
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   orders_key      9799 non-null   int64  
 1   order_id        9799 non-null   str    
 2   shipping_days   9799 non-null   int64  
 3   sales           9799 non-null   float64
 4   order_date_key  9799 non-null   int64  
 5   ship_date_key   9799 non-null   int64  
 6   product_key     9799 non-null   int64  
 7   customer_key    9799 non-null   int64  
 8   location_key    9799 non-null   int64  
 9   ship_mode_key   9799 non-null   int64  
dtypes: float64(1), int64(8), str(1)
memory usage: 900.8 KB


#### Miscelleanous

##### Rearranging column order

In [147]:
products_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 1849 entries, 0 to 1848
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   product_key          1849 non-null   int64   
 1   product_id           1849 non-null   str     
 2   product_name         1849 non-null   str     
 3   category             1849 non-null   str     
 4   sub_category         1849 non-null   str     
 5   product_order_count  1849 non-null   int64   
 6   product_sales        1849 non-null   float64 
 7   order_count_cat      1849 non-null   category
 8   sales_cat            1849 non-null   category
 9   product_value        1849 non-null   str     
dtypes: category(2), float64(1), int64(2), str(5)
memory usage: 265.0 KB


In [148]:
products_dim.columns

Index(['product_key', 'product_id', 'product_name', 'category', 'sub_category',
       'product_order_count', 'product_sales', 'order_count_cat', 'sales_cat',
       'product_value'],
      dtype='str')

In [149]:
products_dim = products_dim[['product_key',  'product_id', 'product_name', 'category', 'sub_category',
                             'product_order_count', 'order_count_cat', 'product_sales', 'sales_cat', 'product_value']]

In [150]:
products_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 1849 entries, 0 to 1848
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   product_key          1849 non-null   int64   
 1   product_id           1849 non-null   str     
 2   product_name         1849 non-null   str     
 3   category             1849 non-null   str     
 4   sub_category         1849 non-null   str     
 5   product_order_count  1849 non-null   int64   
 6   order_count_cat      1849 non-null   category
 7   product_sales        1849 non-null   float64 
 8   sales_cat            1849 non-null   category
 9   product_value        1849 non-null   str     
dtypes: category(2), float64(1), int64(2), str(5)
memory usage: 265.0 KB


In [151]:
orders_fact.info()

<class 'pandas.DataFrame'>
RangeIndex: 9799 entries, 0 to 9798
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   orders_key      9799 non-null   int64  
 1   order_id        9799 non-null   str    
 2   shipping_days   9799 non-null   int64  
 3   sales           9799 non-null   float64
 4   order_date_key  9799 non-null   int64  
 5   ship_date_key   9799 non-null   int64  
 6   product_key     9799 non-null   int64  
 7   customer_key    9799 non-null   int64  
 8   location_key    9799 non-null   int64  
 9   ship_mode_key   9799 non-null   int64  
dtypes: float64(1), int64(8), str(1)
memory usage: 900.8 KB


In [152]:
orders_fact = orders_fact[['orders_key', 'order_id', 'order_date_key', 'ship_date_key', 
                           'ship_mode_key', 'product_key', 'customer_key', 'location_key', 
                           'shipping_days', 'sales']]

In [153]:
df.columns

Index(['order_date', 'ship_date', 'customer_id', 'customer_name', 'segment',
       'country', 'city', 'state', 'postal_code', 'region', 'category',
       'sub_category', 'product_name', 'sales', 'shipping_days', 'ship_mode',
       'order_id', 'product_id'],
      dtype='str')

In [154]:
df.insert(0, 'row_id', range(1, len(df) + 1))

In [155]:
df.drop(columns='shipping_days', inplace=True)

In [156]:
df = df[['row_id', 'order_date', 'order_id', 'ship_date', 'ship_mode', 
         'customer_id', 'customer_name', 'segment', 'country', 'region', 
         'state', 'city', 'postal_code', 'product_id', 'product_name', 
         'category', 'sub_category', 'sales'
         ]]

In [157]:
customers_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 793 entries, 0 to 792
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   customer_key          793 non-null    int64   
 1   customer_id           793 non-null    str     
 2   customer_name         793 non-null    str     
 3   segment               793 non-null    str     
 4   customer_order_count  793 non-null    int64   
 5   customer_sales        793 non-null    float64 
 6   avg_order_value       793 non-null    float64 
 7   order_count_cat       793 non-null    category
 8   sales_cat             793 non-null    category
 9   avg_order_val_cat     793 non-null    category
 10  customer_value        793 non-null    str     
dtypes: category(3), float64(2), int64(2), str(4)
memory usage: 79.2 KB


In [158]:
customers_dim.columns

Index(['customer_key', 'customer_id', 'customer_name', 'segment',
       'customer_order_count', 'customer_sales', 'avg_order_value',
       'order_count_cat', 'sales_cat', 'avg_order_val_cat', 'customer_value'],
      dtype='str')

In [159]:
customers_dim = customers_dim[['customer_key', 'customer_id', 'customer_name', 'segment', 
                               'customer_order_count', 'order_count_cat', 'customer_sales', 'sales_cat', 
                               'avg_order_value', 'avg_order_val_cat', 'customer_value']]

In [160]:
location_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 628 entries, 0 to 627
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   location_key      628 non-null    int64   
 1   country           628 non-null    str     
 2   region            628 non-null    str     
 3   state             628 non-null    str     
 4   city              628 non-null    str     
 5   postal_code       628 non-null    uint32  
 6   total_orders      628 non-null    int64   
 7   total_sales       628 non-null    float64 
 8   total_orders_cat  628 non-null    category
 9   total_sales_cat   628 non-null    category
 10  location_value    628 non-null    str     
dtypes: category(2), float64(1), int64(2), str(5), uint32(1)
memory usage: 70.1 KB


In [161]:
location_dim.columns

Index(['location_key', 'country', 'region', 'state', 'city', 'postal_code',
       'total_orders', 'total_sales', 'total_orders_cat', 'total_sales_cat',
       'location_value'],
      dtype='str')

In [162]:
location_dim.rename(columns={
    'total_orders' : 'location_order_count',
    'total_orders_cat' : 'order_count_cat',
    'total_sales' : 'location_sales',
    'total_sales_cat' : 'sales_cat'
}, inplace=True)

In [163]:
location_dim.columns

Index(['location_key', 'country', 'region', 'state', 'city', 'postal_code',
       'location_order_count', 'location_sales', 'order_count_cat',
       'sales_cat', 'location_value'],
      dtype='str')

In [164]:
location_dim = location_dim[['location_key', 'country', 'region', 'state', 'city', 'postal_code', 
                             'location_order_count', 'order_count_cat', 'location_sales', 'sales_cat', 'location_value']]

In [165]:
location_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 628 entries, 0 to 627
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   location_key          628 non-null    int64   
 1   country               628 non-null    str     
 2   region                628 non-null    str     
 3   state                 628 non-null    str     
 4   city                  628 non-null    str     
 5   postal_code           628 non-null    uint32  
 6   location_order_count  628 non-null    int64   
 7   order_count_cat       628 non-null    category
 8   location_sales        628 non-null    float64 
 9   sales_cat             628 non-null    category
 10  location_value        628 non-null    str     
dtypes: category(2), float64(1), int64(2), str(5), uint32(1)
memory usage: 70.1 KB


##### Downcasting Column dtypes

Orders Fact Table

In [166]:
# Float columns

for col in orders_fact.select_dtypes(include=['float']).columns:
    orders_fact[col] = pd.to_numeric(orders_fact[col], downcast='float')

# Integer columns

for col in orders_fact.select_dtypes(include=['int']).columns:
    if (orders_fact[col] >= 0).all():   # only non-negative values
        orders_fact[col] = pd.to_numeric(orders_fact[col], downcast='unsigned')
    else:
        orders_fact[col] = pd.to_numeric(orders_fact[col], downcast='integer')

# Str columns

orders_fact[orders_fact.select_dtypes(include='str').columns] = orders_fact.select_dtypes(include='str').astype('category')

Product Dimension Table

In [167]:
# Integer columns

for col in products_dim.select_dtypes(include=['int']).columns:
    if (products_dim[col] >= 0).all():   # only non-negative values
        products_dim[col] = pd.to_numeric(products_dim[col], downcast='unsigned')
    else:
        products_dim[col] = pd.to_numeric(products_dim[col], downcast='integer')

# Str columns

products_dim[products_dim.select_dtypes(include='str').columns] = products_dim.select_dtypes(include='str').astype('category')

Date Dimension Table

In [168]:
# Integer columns
for col in date_dim.select_dtypes(include=['int']).columns:
    if (date_dim[col] >= 0).all():   # only non-negative values
        date_dim[col] = pd.to_numeric(date_dim[col], downcast='unsigned')
    else:
        date_dim[col] = pd.to_numeric(date_dim[col], downcast='integer')

# Str columns

date_dim[date_dim.select_dtypes(include='str').columns] = date_dim.select_dtypes(include='str').astype('category')

Customer Dimension Table

In [169]:
# Integer columns

for col in customers_dim.select_dtypes(include=['int']).columns:
    if (customers_dim[col] >= 0).all():   # only non-negative values
        customers_dim[col] = pd.to_numeric(customers_dim[col], downcast='unsigned')
    else:
        customers_dim[col] = pd.to_numeric(customers_dim[col], downcast='integer')

# Str columns

customers_dim[customers_dim.select_dtypes(include='str').columns] = customers_dim.select_dtypes(include='str').astype('category')

Location Dimension Table

In [170]:
# Integer columns

for col in location_dim.select_dtypes(include=['int']).columns:
    if (location_dim[col] >= 0).all():   # only non-negative values
        location_dim[col] = pd.to_numeric(location_dim[col], downcast='unsigned')
    else:
        location_dim[col] = pd.to_numeric(location_dim[col], downcast='integer')

# Str columns

location_dim[location_dim.select_dtypes(include='str').columns] = location_dim.select_dtypes(include='str').astype('category')

Ship Mode Dimension Table

In [171]:
# Integer columns

for col in ship_mode_dim.select_dtypes(include=['int']).columns:
    if (ship_mode_dim[col] >= 0).all():   # only non-negative values
        ship_mode_dim[col] = pd.to_numeric(ship_mode_dim[col], downcast='unsigned')
    else:
        ship_mode_dim[col] = pd.to_numeric(ship_mode_dim[col], downcast='integer')

# Str columns

ship_mode_dim[ship_mode_dim.select_dtypes(include='str').columns] = ship_mode_dim.select_dtypes(include='str').astype('category')

df Table

In [172]:
# Float columns

for col in df.select_dtypes(include=['float']).columns:
    df[col] = pd.to_numeric(df[col], downcast='float')

# Integer columns

for col in df.select_dtypes(include=['int']).columns:
    if (df[col] >= 0).all():   # only non-negative values
        df[col] = pd.to_numeric(df[col], downcast='unsigned')
    else:
        df[col] = pd.to_numeric(df[col], downcast='integer')

# Str columns

df[df.select_dtypes(include='str').columns] = df.select_dtypes(include='str').astype('category')

### Load

In [173]:
orders_fact.to_parquet('/app/data/processed/orders_fact.parquet', engine='pyarrow', index=False)

date_dim.to_parquet('/app/data/processed/date_dim.parquet', engine='pyarrow', index=False)
customers_dim.to_parquet('/app/data/processed/customers_dim.parquet', engine='pyarrow', index=False)
products_dim.to_parquet('/app/data/processed/products_dim.parquet', engine='pyarrow', index=False)
location_dim.to_parquet('/app/data/processed/location_dim.parquet', engine='pyarrow', index=False)
ship_mode_dim.to_parquet('/app/data/processed/ship_mode_dim.parquet', engine='pyarrow', index=False)

df.to_parquet('/app/data/processed/us_superstore_sales.parquet', engine='pyarrow', index=False)